# TNG tour
A hands-on tour of the TNG module (`galactic_tidal_constraints.tng`): it reads the IllustrisTNG (TNG50-1) data this
project already has, exactly as stored, with no unit changes and no change of frame. Run the cells in order.

- `tng.read_table(path)`: the target table, one array per column
- `tng.read_archive(path, particle_types)`: one target's particles

The files live in the data folder, outside the repo. The tour looks for it in `GTC_DATA_DIR`, or else in `data/`
next to the repo. Design: `docs/modules/tng.md`.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints import tng

DATA = Path(os.environ.get("GTC_DATA_DIR", "../../../data")) / "tng-legacy-data"
TARGET = 62000000190717

## The target table
`read_table` returns a plain dict: column name → array, one entry per row.

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")

print(type(table).__name__, "with", len(table), "columns,", len(table["snap"]), "rows")
print("encounters:", len(np.unique(table["moi_2"])))

kinds = {}
for name, column in table.items():
    kinds.setdefault((column.dtype.kind, column.shape[1:]), []).append(name)
for (kind, shape), names in kinds.items():
    print()
    print(f"{len(names):3d} columns of dtype kind {kind!r}, shape (rows, *{shape}):")
    print("    " + ", ".join(names[:6]) + (" ..." if len(names) > 6 else ""))

### One target's row
IDs are integers, so a full `SubhaloIDRaw` matches exactly. The catalog values are raw TNG units (ckpc/h, km/s, 10¹⁰ M☉/h).

In [ ]:
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
for name in ("moi_2", "snap", "p_SubhaloIDRaw", "s_SubhaloIDRaw", "redshift", "scale_factor", "time_gyr",
             "p_SubhaloMass", "s_SubhaloMass", "p_SubhaloPos", "s_SubhaloPos", "p_SubhaloVel", "p_SubhaloSpin"):
    print(f"{name:16s} {table[name][row]}")
print()
print("orbital_frame_pos_transform:")
print(np.round(table["orbital_frame_pos_transform"][row], 3))

### One encounter's rows
All snapshots of the same encounter share `moi_2`.

In [ ]:
rows = np.flatnonzero(table["moi_2"] == table["moi_2"][row])
rows = rows[np.argsort(table["snap"][rows])]
print("snapshots:", table["snap"][rows].tolist())
print("time (Gyr): %.2f to %.2f" % (table["time_gyr"][rows][0], table["time_gyr"][rows][-1]))

separation = np.linalg.norm(table["s_SubhaloPos"][rows] - table["p_SubhaloPos"][rows], axis=1)
plt.figure(figsize=(6, 3))
plt.plot(table["time_gyr"][rows], separation, "o-")
plt.axvline(table["time_gyr"][row], color="gray", ls=":", label="target snapshot")
plt.xlabel("age of the universe (Gyr)"); plt.ylabel("separation (ckpc/h, as stored)"); plt.legend();

## A target archive
Stars by default. The values are raw TNG snapshot units, in the earlier pipeline's orbital frame.

In [ ]:
stars = tng.read_archive(DATA / f"target-archive-{TARGET}.h5")
for name, value in stars.items():
    print(f"{name:18s}", f"{value.shape} {value.dtype}" if isinstance(value, np.ndarray) else value)

In [ ]:
for name in ("subfind_galaxy", "historical_galaxy"):
    print(f"{name:18s} primary {int((stars[name] == 0).sum()):,}   secondary {int((stars[name] == 1).sum()):,}")
moved = int((stars["subfind_galaxy"] != stars["historical_galaxy"]).sum())
print(f"stars whose historical galaxy differs from SUBFIND's: {moved:,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
for ax, galaxy, title in zip(axes, (0, 1), ("primary", "secondary")):
    xy = stars["positions"][stars["historical_galaxy"] == galaxy, :2]
    ax.hist2d(xy[:, 0], xy[:, 1], bins=300, range=[[-60, 60], [-60, 60]], norm="log", cmap="gray")
    ax.set_title(f"{title} stars (historical label)"); ax.set_aspect("equal"); ax.set_xlabel("x (ckpc/h, as stored)")
axes[0].set_ylabel("y (ckpc/h, as stored)");

### Other particle types
One name, several, or `"all"`.

In [ ]:
everything = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
for name, number in tng.PARTICLE_TYPES.items():
    kept = everything["particle_type"] == number
    total = everything["masses"][kept].sum(dtype=np.float64)
    print(f"{name:6s} {int(kept.sum()):>10,}   total mass {total:10.3f}  (1e10 Msun/h)")

dm = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "dm")
print()
print("dark matter particle mass:", np.unique(dm["masses"]), "=", f"{dm['masses'][0] * 1e10 / 0.6774:.3e} Msun (TNG50-1: 4.5e5)")

## Measuring a galaxy
`galaxy_properties` takes plain arrays and returns the mass, center of mass, mean velocity and spin (angular
momentum per unit mass). Nothing is converted: the results are in the units of the arrays given.

In [ ]:
archive = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
measured = {}
for galaxy, side in ((0, "p"), (1, "s")):
    kept = archive["subfind_galaxy"] == galaxy
    measured[side] = tng.galaxy_properties(archive["positions"][kept], archive["velocities"][kept], archive["masses"][kept])

for name, value in measured["p"].items():
    print(f"{name:15s} {type(value).__name__:8s} {value}")

### Against the TNG catalog
The same two galaxies in the catalog (the table's `p_` and `s_` columns). With a the snapshot's scale factor:
velocities need √a, and the catalog's spin is the particles' spin × a^1.5 (a physical length and a physical velocity).

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
a = table["scale_factor"][row]
size = np.linalg.norm

print(f"a = {a:.4f}\n")
print(f"{'':32s}{'particles':>12s}{'catalog':>12s}{'ratio':>10s}")
def line(label, mine, catalog):
    print(f"{label:32s}{mine:12.4f}{catalog:12.4f}{mine / catalog:10.5f}")

for side, name in (("p", "primary"), ("s", "secondary")):
    spin, catalog_spin = size(measured[side]["spin"]), size(table[f"{side}_SubhaloSpin"][row])
    line(f"{name} mass", measured[side]["mass"], table[f"{side}_SubhaloMass"][row])
    line(f"{name} spin, as stored", spin, catalog_spin)
    line(f"{name} spin x a^1.5", spin * a**1.5, catalog_spin)
    line(f"{name} spin x mass (the old way)", spin * measured[side]["mass"], catalog_spin)

separation = size(measured["p"]["center_of_mass"] - measured["s"]["center_of_mass"])
speed = size(measured["p"]["velocity"] - measured["s"]["velocity"])
line("separation of the centers", separation, size(table["p_SubhaloCM"][row] - table["s_SubhaloCM"][row]))
line("relative speed, as stored", speed, size(table["p_SubhaloVel"][row] - table["s_SubhaloVel"][row]))
line("relative speed x sqrt(a)", speed * np.sqrt(a), size(table["p_SubhaloVel"][row] - table["s_SubhaloVel"][row]))

### Through the old frame
The archive's particles are in the earlier pipeline's frame: the box moved by the table's two 4×4 matrices (a rotation
and a shift). Positions and velocities take the whole matrix; a spin takes the rotation only.

In [ ]:
pos_matrix, vel_matrix = table["orbital_frame_pos_transform"][row], table["orbital_frame_vel_transform"][row]
rotation = vel_matrix[:3, :3]
move = lambda matrix, vector: matrix[:3, :3] @ vector + matrix[:3, 3]
angle = lambda u, v: np.degrees(np.arccos(np.clip(u @ v / size(u) / size(v), -1, 1)))

print("the two matrices share one rotation:", np.array_equal(pos_matrix[:3, :3], rotation),
      "  det =", round(float(np.linalg.det(rotation)), 6))
for side, name in (("p", "primary"), ("s", "secondary")):
    center = move(pos_matrix, table[f"{side}_SubhaloCM"][row])
    velocity = move(vel_matrix, table[f"{side}_SubhaloVel"][row] / np.sqrt(a))
    catalog_spin = table[f"{side}_SubhaloSpin"][row]
    print(f"\n{name}")
    print(f"  center of mass    particles {np.round(measured[side]['center_of_mass'], 3)}   catalog, moved {np.round(center, 3)}")
    print(f"  velocity          particles {np.round(measured[side]['velocity'], 3)}   catalog / sqrt(a), moved {np.round(velocity, 3)}")
    print(f"  spin direction    off by {angle(measured[side]['spin'], rotation @ catalog_spin):.4f} deg with the rotation only,"
          f" {angle(measured[side]['spin'], move(vel_matrix, catalog_spin)):.1f} deg with the shift too (the old way)")

### Half-mass radius
`half_mass_radius` measures from the center it is given. The catalog measures from `SubhaloPos` (the most bound
particle), not from the center of mass.

In [ ]:
for galaxy, side, name in ((0, "p", "primary"), (1, "s", "secondary")):
    kept = archive["subfind_galaxy"] == galaxy
    most_bound = move(pos_matrix, table[f"{side}_SubhaloPos"][row])
    all_types = tng.half_mass_radius(archive["positions"][kept], archive["masses"][kept], most_bound)
    stars = kept & (archive["particle_type"] == tng.PARTICLE_TYPES["stars"])
    stellar = tng.half_mass_radius(archive["positions"][stars], archive["masses"][stars], most_bound)
    print(f"{name:10s} all types {all_types:8.4f}  catalog {table[f'{side}_SubhaloHalfmassRad'][row]:8.4f}"
          f"   stars only {stellar:7.4f}   (ckpc/h, as stored)")

## The box edge
The simulation box (35,000 ckpc/h on a side) wraps around: what leaves through one face comes back in through the
opposite one. A pair on either side of a face looks almost a whole box apart. `nearest_image` moves positions to the
copy nearest a reference point.

In [ ]:
print("box side:", tng.BOX, "ckpc/h")
print(tng.nearest_image([50.0, 100.0, 200.0], [34950.0, 120.0, 180.0]))

### A real encounter that crosses a face
Target `87000000451863`'s encounter: at one snapshot the two galaxies are on either side of a face. With one reference
per row, the whole track is mended in one call.

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")
edge_row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == 87000000451863)[0])
rows = np.flatnonzero(table["moi_2"] == table["moi_2"][edge_row])
rows = rows[np.argsort(table["snap"][rows])]
primary, secondary = table["p_SubhaloPos"][rows], table["s_SubhaloPos"][rows]

stored = np.linalg.norm(secondary - primary, axis=1)
mended = np.linalg.norm(tng.nearest_image(secondary, primary) - primary, axis=1)
worst = int(np.argmax(stored))
print("snapshot", table["snap"][rows][worst])
print("  primary  ", primary[worst])
print("  secondary", secondary[worst], "-> mended", np.round(tng.nearest_image(secondary[worst], primary[worst]), 1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True)
axes[0].plot(table["snap"][rows], stored, "o-"); axes[0].set_title("as stored")
axes[1].plot(table["snap"][rows], mended, "o-"); axes[1].set_title("with nearest_image")
for ax in axes:
    ax.axvline(table["snap"][edge_row], color="gray", ls=":"); ax.set_xlabel("snapshot")
axes[0].set_ylabel("separation (ckpc/h)");

## Out of the old frame
An archive's particles are stored in the earlier pipeline's frame (the box, rotated and shifted). `archive_to_box`
undoes that with the table's two matrices, giving box-frame positions and velocities in stored units. There the
particles line up with the catalog directly.

In [ ]:
archive = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
positions, velocities = tng.archive_to_box(archive["positions"], archive["velocities"],
                                           table["orbital_frame_pos_transform"][row], table["orbital_frame_vel_transform"][row])

print("as read (old frame):", np.round(archive["positions"].min(axis=0)), "to", np.round(archive["positions"].max(axis=0)))
print("in the box frame:   ", np.round(positions.min(axis=0)), "to", np.round(positions.max(axis=0)))
for galaxy, side, name in ((0, "p", "primary"), (1, "s", "secondary")):
    kept = archive["subfind_galaxy"] == galaxy
    nucleus = table[f"{side}_SubhaloPos"][row]
    measured_box = tng.galaxy_properties(positions[kept], velocities[kept], archive["masses"][kept])
    print(f"\n{name}")
    print(f"  catalog SubhaloPos {nucleus}   nearest particle {np.linalg.norm(positions[kept] - nucleus, axis=1).min():.4f} ckpc/h away")
    print(f"  catalog SubhaloCM  {table[f'{side}_SubhaloCM'][row]}   from particles {np.round(measured_box['center_of_mass'], 3)}")

### A galaxy split by the box edge
Target `87000000451863`'s primary sits across a face: in the box frame a few of its particles are at the far side.
`nearest_image` around its nucleus brings them back. (Target `49000000112167` is the large case: 47,690 particles.)

In [ ]:
SPLIT = 87000000451863
split = tng.read_archive(DATA / f"target-archive-{SPLIT}.h5", "all")
split_row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == SPLIT)[0])
primary = split["subfind_galaxy"] == 0
stored, _ = tng.archive_to_box(split["positions"][primary], split["velocities"][primary],
                               table["orbital_frame_pos_transform"][split_row], table["orbital_frame_vel_transform"][split_row])
nucleus = table["p_SubhaloPos"][split_row]
mended = tng.nearest_image(stored, nucleus)

print("nucleus (SubhaloPos):", nucleus)
print("extent as stored:    ", np.round(np.ptp(stored, axis=0)), "ckpc/h")
print("extent mended:       ", np.round(np.ptp(mended, axis=0)), "ckpc/h")
moved = np.abs(mended - stored).max(axis=1) > 1
print("particles moved:     ", int(moved.sum()), "of", f"{len(stored):,}")
print("their z, stored:     ", np.round(stored[moved, 2], 1))
print("their z, mended:     ", np.round(mended[moved, 2], 1))

## Units
Two explicit steps. `physical` turns stored TNG values into kpc, km/s and M☉, using TNG's h and the snapshot's scale
factor. `spam_units` turns those into SPAM's units (15 kpc, 10¹¹ M☉, and the velocity and time that follow with G = 1).

In [ ]:
print(f"TNG's h            {tng.H}")
print(f"SPAM's length      {tng.SPAM_LENGTH} kpc")
print(f"SPAM's mass        {tng.SPAM_MASS:.0e} Msun")
print(f"SPAM's velocity    {tng.SPAM_VELOCITY:.3f} km/s")
print(f"SPAM's time        {tng.SPAM_TIME * 1000:.2f} Myr")

### One target's catalog values
The quantity's name picks the factor. Lengths and particle velocities need the scale factor; masses, catalog
velocities and the catalog's spin don't.

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
a = table["scale_factor"][row]

separation = tng.physical(table["s_SubhaloPos"][row] - table["p_SubhaloPos"][row], "length", a)
speed = tng.physical(table["s_SubhaloVel"][row] - table["p_SubhaloVel"][row], "velocity")
masses = tng.physical([table["p_SubhaloMass"][row], table["s_SubhaloMass"][row]], "mass")
radii = tng.physical([table["p_SubhaloHalfmassRad"][row], table["s_SubhaloHalfmassRad"][row]], "length", a)

print(f"a = {a:.4f}\n")
print(f"{'':28s}{'stored':>12s}{'physical':>16s}{'SPAM units':>12s}")
def line(label, stored, value, unit, quantity):
    print(f"{label:28s}{stored:12.4f}{value:12.4g} {unit:5s}{tng.spam_units(value, quantity):10.4f}")

line("separation", np.linalg.norm(table["s_SubhaloPos"][row] - table["p_SubhaloPos"][row]), np.linalg.norm(separation), "kpc", "length")
line("relative speed", np.linalg.norm(speed), np.linalg.norm(speed), "km/s", "velocity")
line("primary mass", table["p_SubhaloMass"][row], masses[0], "Msun", "mass")
line("secondary mass", table["s_SubhaloMass"][row], masses[1], "Msun", "mass")
line("primary half-mass radius", table["p_SubhaloHalfmassRad"][row], radii[0], "kpc", "length")
line("secondary half-mass radius", table["s_SubhaloHalfmassRad"][row], radii[1], "kpc", "length")

### A whole column, and particles
One scale factor per row converts a table column in one call. Particles converted first can be measured directly in
physical units: the spin then comes out in kpc km/s and matches the catalog's, converted.

In [ ]:
rows = np.flatnonzero(table["moi_2"] == table["moi_2"][row])
rows = rows[np.argsort(table["snap"][rows])]
apart = tng.physical(table["s_SubhaloPos"][rows] - table["p_SubhaloPos"][rows], "length", table["scale_factor"][rows])
print("separation at every snapshot of the encounter (kpc):")
print(np.round(np.linalg.norm(apart, axis=1), 1))

archive = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
primary = archive["subfind_galaxy"] == 0
galaxy = tng.galaxy_properties(tng.physical(archive["positions"][primary], "length", a),
                               tng.physical(archive["velocities"][primary], "particle_velocity", a),
                               tng.physical(archive["masses"][primary], "mass"))
print()
print(f"primary from converted particles:  mass {galaxy['mass']:.4e} Msun   spin {np.linalg.norm(galaxy['spin']):.1f} kpc km/s")
print(f"primary from the catalog:          mass {masses[0]:.4e} Msun   spin {np.linalg.norm(tng.physical(table['p_SubhaloSpin'][row], 'spin')):.1f} kpc km/s")

## The frame SPAM uses
SPAM works with the primary's nucleus at the origin and at rest. The steps for one target, in order:
out of the old frame, whole across the box edge, physical units, each nucleus's velocity, relative to the primary,
then one rotation that puts the secondary on +x and the orbit in the x-y plane.

In [ ]:
def spam_frame(target, table, radius=1.0, hubble_flow=True):
    """One target's stars in the frame SPAM uses, and what was measured on the way."""
    row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target)[0])
    a, p_pos, s_pos = table["scale_factor"][row], table["p_SubhaloPos"][row], table["s_SubhaloPos"][row]
    archive = tng.read_archive(DATA / f"target-archive-{target}.h5", ("stars", "bh"))
    positions, velocities = tng.archive_to_box(archive["positions"], archive["velocities"],
                                               table["orbital_frame_pos_transform"][row], table["orbital_frame_vel_transform"][row])
    positions = tng.physical(tng.nearest_image(positions, p_pos), "length", a)      # kpc
    velocities = tng.physical(velocities, "particle_velocity", a)                   # km/s
    nucleus = {"p": tng.physical(p_pos, "length", a), "s": tng.physical(tng.nearest_image(s_pos, p_pos), "length", a)}

    stars = archive["particle_type"] == tng.PARTICLE_TYPES["stars"]
    star_velocity, hole_velocity = {}, {}
    for galaxy, side in ((0, "p"), (1, "s")):
        star_velocity[side] = tng.core_velocity(positions[stars], velocities[stars], archive["masses"][stars], nucleus[side], radius)
        hole_velocity[side] = velocities[~stars & (archive["subfind_galaxy"] == galaxy)][0]

    secondary = tng.relative_to(nucleus["s"], star_velocity["s"], nucleus["p"], star_velocity["p"], a, hubble_flow)
    R = tng.orbit_rotation(*secondary)
    positions, velocities = tng.relative_to(positions[stars], velocities[stars], nucleus["p"], star_velocity["p"], a, hubble_flow)
    return {"row": row, "a": a, "R": R, "nucleus": nucleus, "star_velocity": star_velocity, "hole_velocity": hole_velocity,
            "secondary": (secondary[0] @ R.T, secondary[1] @ R.T), "positions": positions @ R.T, "velocities": velocities @ R.T,
            "galaxy": archive["historical_galaxy"][stars]}

table = tng.read_table(DATA / "moi-4-targets.h5")
frame = spam_frame(TARGET, table)
print("rotation R (rows are the new x, y, z axes):")
print(np.round(frame["R"], 3))
print("secondary's nucleus, kpc: ", np.round(frame["secondary"][0], 3))
print("secondary's velocity, km/s:", np.round(frame["secondary"][1], 2))
print("Hubble rate:", round(float(tng.hubble_rate(frame["a"])), 4), "km/s per kpc")

### Three ways to measure how fast the pair is moving
The catalog's `SubhaloVel` averages each whole halo. The nucleus moves differently: measured from the stars within
1 kpc of it, and from its black hole. (Relative speed of the two galaxies, with the Hubble flow.)

In [ ]:
frames = {target: spam_frame(target, table) for target in (62000000190717, 75000000121160, 75000000317209)}
print(f"{'target':16s}{'apart':>9s}{'catalog':>10s}{'stars<1kpc':>12s}{'black holes':>13s}{'stars vs holes':>16s}{'plane: catalog':>16s}{'holes':>8s}")
for target, f in frames.items():
    row, a, nucleus = f["row"], f["a"], f["nucleus"]
    relative = {"stars": tng.relative_to(nucleus["s"], f["star_velocity"]["s"], nucleus["p"], f["star_velocity"]["p"], a),
                "holes": tng.relative_to(nucleus["s"], f["hole_velocity"]["s"], nucleus["p"], f["hole_velocity"]["p"], a),
                "catalog": tng.relative_to(nucleus["s"], table["s_SubhaloVel"][row], nucleus["p"], table["p_SubhaloVel"][row], a)}
    speed = {name: np.linalg.norm(value[1]) for name, value in relative.items()}
    normal = {name: tng.orbit_rotation(*value)[2] for name, value in relative.items()}
    tilt = {name: np.degrees(np.arccos(np.clip(normal[name] @ normal["stars"], -1, 1))) for name in ("catalog", "holes")}
    print(f"{target:<16d}{np.linalg.norm(relative['stars'][0]):6.1f} kpc{speed['catalog']:7.0f} km/s{speed['stars']:8.0f} km/s"
          f"{speed['holes']:9.0f} km/s{np.linalg.norm(relative['stars'][1] - relative['holes'][1]):11.1f} km/s"
          f"{tilt['catalog']:12.1f} deg{tilt['holes']:6.1f} deg")
print()
print("'stars vs holes' is the size of the difference between the two velocity vectors;")
print("'plane' is how far each orbit plane is from the plane the stars give.")

### The three targets, face-on
Stars in the new frame: the primary's nucleus at the origin, the secondary's on +x (marked), the pair turning
counter-clockwise. Left: the orbit plane (x-y). Right: edge-on (x-z).

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
for (target, f), (top, side) in zip(frames.items(), axes):
    apart = f["secondary"][0][0]
    half = max(1.6 * apart, 40)
    for ax, (i, j), name in ((top, (0, 1), "x-y: the orbit plane"), (side, (0, 2), "x-z: edge-on")):
        ax.hist2d(f["positions"][:, i], f["positions"][:, j], bins=400, range=[[-half, half], [-half, half]], norm="log", cmap="gray")
        ax.plot([0, apart], [0, 0], "+", color="tab:red", ms=10)
        ax.set_aspect("equal"); ax.set_title(f"{target}   {name}"); ax.set_xlabel("x (kpc)"); ax.set_ylabel("xyz"[j] + " (kpc)")
    top.arrow(apart, 0, *(0.1 * f["secondary"][1][:2]), color="tab:red", width=0.01 * half, length_includes_head=True)
plt.tight_layout();

### The earlier orbit in this frame
The table has the two nuclei's positions at every snapshot. Turned by the same rotation, they show how well the plane
of the target snapshot fits the earlier orbit. (Only positions: the nucleus velocity is measured at the target
snapshot alone.) Points are snapshots up to the target; the large dot is the target snapshot.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 13))
for (target, f), (top, side) in zip(frames.items(), axes):
    rows = np.flatnonzero(table["moi_2"] == table["moi_2"][f["row"]])
    rows = rows[np.argsort(table["snap"][rows])]
    rows = rows[table["snap"][rows] <= table["snap"][f["row"]]]
    apart = tng.nearest_image(table["s_SubhaloPos"][rows], table["p_SubhaloPos"][rows]) - table["p_SubhaloPos"][rows]
    track = tng.physical(apart, "length", table["scale_factor"][rows]) @ f["R"].T
    ago = table["time_gyr"][f["row"]] - table["time_gyr"][rows]
    recent = ago <= 1.5
    for ax, j, name in ((top, 1, "x-y: the orbit plane"), (side, 2, "x-z: edge-on")):
        ax.plot(track[:, 0], track[:, j], "-", color="lightgray")
        points = ax.scatter(track[:, 0], track[:, j], c=ago, cmap="viridis_r", s=18)
        ax.plot(track[-1, 0], track[-1, j], "o", color="tab:red", ms=9); ax.plot(0, 0, "k+", ms=12)
        ax.set_aspect("equal", adjustable="datalim"); ax.set_title(f"{target}   {name}")
        ax.set_xlabel("x (kpc)"); ax.set_ylabel("xyz"[j] + " (kpc)")
    plt.colorbar(points, ax=side, label="Gyr before the target snapshot")
    off = np.abs(track[recent, 2]) / np.linalg.norm(track[recent], axis=1)
    print(f"{target}: over the last 1.5 Gyr the track is at most {off.max():.0%} of its distance out of the plane")
plt.tight_layout();

## Disk angles
SPAM tilts each disk with two angles, φ and θ. `disk_angles` gets them from a spin direction in SPAM's frame:
θ is the angle from +z (0 is flat and turning with the orbit, 180 flat and against it, 90 edge-on) and φ is the
direction of the spin's part in the plane.

In [ ]:
for spin in ([0, 0, 1], [0, 0, -1], [1, 0, 0], [0, 1, 0], [1, 1, 1]):
    phi, theta = tng.disk_angles(spin)
    print(f"spin along {str(spin):12s} ->  phi {phi:6.1f}   theta {theta:6.1f}")

### The three targets' disks
Each galaxy's spin here is measured from the stars it owned before the encounter (the historical label), in the
frame from the last section. Which stars should define a disk's spin is a choice for the target builder; this only
shows the function at work.

In [ ]:
if "frames" not in globals():
    table = tng.read_table(DATA / "moi-4-targets.h5")
    frames = {target: spam_frame(target, table) for target in (62000000190717, 75000000121160, 75000000317209)}

angles = {}
for target, f in frames.items():
    for galaxy, name in ((0, "primary"), (1, "secondary")):
        kept = f["galaxy"] == galaxy
        spin = tng.galaxy_properties(f["positions"][kept], f["velocities"][kept], np.ones(int(kept.sum())))["spin"]
        angles[target, galaxy] = tng.disk_angles(spin)
        print(f"{target}  {name:10s} spin direction {np.round(spin / np.linalg.norm(spin), 3)}   "
              f"phi {angles[target, galaxy][0]:6.1f}   theta {angles[target, galaxy][1]:6.1f}")

### Through SPAM
SPAM builds a disk from those angles (here the primary of the first target, with nothing else matched: not its size
or its mass). Its spin, measured from SPAM's particles, points the same way as the TNG stars' spin did.

In [ ]:
from galactic_tidal_constraints.simr import simulation as sim

target = 62000000190717
model = np.zeros(14)
model[[sim.PARAM_INDEX[name] for name in ("x", "vy", "mass1", "mass2", "rout1", "rout2")]] = [8.0, 0.5, 1.0, 1.0, 1.0, 1.0]
model[sim.PARAM_INDEX["phi1"]], model[sim.PARAM_INDEX["theta1"]] = angles[target, 0]
model[sim.PARAM_INDEX["phi2"]], model[sim.PARAM_INDEX["theta2"]] = angles[target, 1]
disks = sim.unperturbed_particles(model, sim.SimulatorConfig(n1=3000, n2=3000))

f = frames[target]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
for ax, (i, j) in zip(axes, ((0, 1), (0, 2), (1, 2))):
    ax.plot(disks[:3000, i], disks[:3000, j], ",", color="tab:blue")
    ax.set_aspect("equal"); ax.set_xlabel("xyz"[i] + " (SPAM units)"); ax.set_ylabel("xyz"[j] + " (SPAM units)")
axes[0].set_title("SPAM's primary disk from the TNG angles"); axes[1].set_title("x-z"); axes[2].set_title("y-z")
plt.tight_layout()

for number, (galaxy, name) in enumerate(((0, "primary"), (1, "secondary"))):
    disk = disks[number * 3000:(number + 1) * 3000]
    spam_spin = tng.galaxy_properties(disk[:, :3], disk[:, 3:], np.ones(3000))["spin"]
    kept = f["galaxy"] == galaxy
    tng_spin = tng.galaxy_properties(f["positions"][kept], f["velocities"][kept], np.ones(int(kept.sum())))["spin"]
    off = np.degrees(np.arccos(np.clip(spam_spin @ tng_spin / np.linalg.norm(spam_spin) / np.linalg.norm(tng_spin), -1, 1)))
    print(f"{name}: SPAM's disk spins {off:.3f} degrees from the TNG stars' spin")

## Which galaxy is the primary
`primary_side` answers with the table's letter, `"p"` or `"s"`, by a named rule: `"table"` keeps the table's own
label, and `"stars"` picks the galaxy with more mass in the stars it owned before the encounter. Nothing is renamed;
the answer says which galaxy goes at the origin.

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")
for target in (62000000190717, 75000000121160, 75000000317209):
    archive = tng.read_archive(DATA / f"target-archive-{target}.h5")   # stars
    own = {side: archive["masses"][archive["historical_galaxy"] == label].sum(dtype=np.float64)
           for side, label in tng.GALAXY_LABELS.items()}
    print(f"{target}   stars owned: p {own['p']:.2f}, s {own['s']:.2f} (1e10 Msun/h)   "
          f"table rule -> {tng.primary_side(archive, 'table')!r}   stars rule -> {tng.primary_side(archive, 'stars')!r}")

### Why not the catalog's total mass
During an encounter the two galaxies share one dark halo, and SUBFIND gives it to whichever it calls the central one,
so `SubhaloMass` swings. The stars' mass ratio holds steady. (The star masses along the track are the earlier
pipeline's `ps_` and `ss_` columns; at the target snapshot they equal the archive sums above.)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, target in zip(axes, (62000000190717, 75000000121160, 75000000317209)):
    row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target)[0])
    rows = np.flatnonzero(table["moi_2"] == table["moi_2"][row])
    rows = rows[np.argsort(table["snap"][rows])]
    rows = rows[table["snap"][rows] <= table["snap"][row]]
    ago = table["time_gyr"][row] - table["time_gyr"][rows]
    ax.plot(-ago, table["s_SubhaloMass"][rows] / table["p_SubhaloMass"][rows], "o-", label="catalog SubhaloMass")
    ax.plot(-ago, table["ss_SubhaloMass"][rows] / table["ps_SubhaloMass"][rows], "s-", label="stars, by history")
    ax.axhline(1, color="gray", ls=":"); ax.set_yscale("log"); ax.set_title(str(target))
    ax.set_xlabel("Gyr before the target snapshot")
axes[0].set_ylabel("mass ratio, table's s / p"); axes[0].legend();

## Measuring a target
`measure_target(table, row, archive, config)` puts the pieces above together for one target at its snapshot. It
returns one dict in physical units (kpc, km/s, M☉), with every vector in the frame SPAM uses. It needs every particle
type, so the archive is read with `"all"`. The choices it makes are the fields of `TngTargetConfig`.

In [ ]:
config = tng.TngTargetConfig()
print(config)

row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
archive = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
target = tng.measure_target(table, row, archive, config)


def show(value, indent=0):
    """Print a result of measure_target, one entry per line."""
    for name, entry in value.items():
        if isinstance(entry, dict):
            print(" " * indent + f"{name}:")
            show(entry, indent + 4)
        elif isinstance(entry, np.ndarray) and entry.ndim == 2:
            print(" " * indent + f"{name}:")
            print("\n".join(" " * (indent + 4) + str(np.round(r, 4)) for r in entry))
        elif isinstance(entry, np.ndarray):
            print(" " * indent + f"{name}: {np.round(entry, 3)}")
        elif isinstance(entry, float):
            print(" " * indent + f"{name}: {entry:.5g}")
        else:
            print(" " * indent + f"{name}: {entry}")


show(target)

### Three targets, and how the spin depends on which stars are used
By default a disk's spin comes from the galaxy's stars within 5 stellar half-mass radii of its nucleus (the disk
radius). `spin_radius_factor` changes that number, and `None` takes all the stars the galaxy owned before the
encounter. The loop measures each target with all stars, then again with five radius factors. (About a minute: three whole archives are read.)

In [ ]:
TARGETS = (62000000190717, 75000000121160, 75000000317209)
FACTORS = (1.0, 2.0, 3.0, 5.0, 10.0)
measured, by_factor = {}, {}
for target_id in TARGETS:
    row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target_id)[0])
    archive = tng.read_archive(DATA / f"target-archive-{target_id}.h5", "all")
    measured[target_id] = tng.measure_target(table, row, archive, tng.TngTargetConfig(spin_radius_factor=None))
    by_factor[target_id] = {factor: tng.measure_target(table, row, archive, tng.TngTargetConfig(spin_radius_factor=factor))
                            for factor in FACTORS}
del archive

for target_id, t in measured.items():
    print(f"{target_id}   snapshot {t['snap']}   primary is the table's {t['primary_side']!r}   "
          f"nuclei {t['separation']:.1f} kpc apart")
    for name, velocity in t["velocities"].items():
        print(f"    relative velocity by {name:10s} {np.round(velocity, 1)}   speed {np.linalg.norm(velocity):6.1f} km/s")

In [ ]:
print(f"{'target':15s} {'galaxy':10s} {'historical':>11s} {'catalog':>11s} {'in radius':>11s} {'stars':>11s}"
      f" {'half-mass':>10s} {'disk':>7s} {'phi':>7s} {'theta':>7s}")
print(f"{'':26s} {'------------------ Msun ------------------':>47s} {'--- kpc ---':>18s} {'-- degrees --':>15s}")
for target_id, t in measured.items():
    for role in ("primary", "secondary"):
        g = t[role]
        print(f"{target_id:<15d} {role + ' ' + g['side']:10s} {g['masses']['historical']:11.3e} {g['masses']['catalog']:11.3e}"
              f" {g['masses']['within_radius']:11.3e} {g['stellar_mass']:11.3e} {g['half_mass_radius']:10.2f}"
              f" {g['disk_radius']:7.1f} {g['phi']:7.1f} {g['theta']:7.1f}")

In [ ]:
def angle(a, b):
    return np.degrees(np.arccos(np.clip(a @ b / np.linalg.norm(a) / np.linalg.norm(b), -1, 1)))


print("Each disk's (phi, theta), and how far its spin is from the all-stars spin, in degrees:\n")
print(f"{'target':15s} {'galaxy':10s} {'all stars':>16s}" + "".join(f"{'within ' + str(int(f)):>24s}" for f in FACTORS))
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (target_id, t) in zip(axes, measured.items()):
    for role in ("primary", "secondary"):
        g = t[role]
        off = [angle(by_factor[target_id][f][role]["spin"], g["spin"]) for f in FACTORS]
        cells_ = "".join(f"   ({s['phi']:5.1f}, {s['theta']:5.1f}) {o:5.1f}"
                         for s, o in ((by_factor[target_id][f][role], o) for f, o in zip(FACTORS, off)))
        print(f"{target_id:<15d} {role + ' ' + g['side']:10s}   ({g['phi']:5.1f}, {g['theta']:5.1f})" + cells_)
        kept = [by_factor[target_id][f][role]["spin_stars"] / g["spin_stars"] for f in FACTORS]
        ax.plot(FACTORS, off, "o-", label=f"{role} (within 10: {kept[-1]:.0%} of its stars)")
    ax.set_title(str(target_id)); ax.set_xlabel("spin from stars within this many half-mass radii"); ax.legend(fontsize=8)
axes[0].set_ylabel("degrees from the all-stars spin");

## SPAM's 14 parameters
`spam_model(target)` turns a measured target into the model SPAM runs: 14 numbers in SPAM's order and units
(15 kpc, about 169.3 km/s, 10¹¹ M☉; angles in degrees). It measures and chooses nothing.

In [ ]:
model = tng.spam_model(target)          # `target`: 62000000190717 measured with the default config, above
print(type(model).__name__, model.shape, model.dtype)
for name, value in zip(tng.SPAM_PARAMETERS, model):
    print(f"    {name:7s} {value:10.4f}")

SPAM takes it as it is. It integrates the secondary back in time from these values and then forward again with the
disks' particles, so its secondary ends where the target says it is now (the last row of the particles).

In [ ]:
from galactic_tidal_constraints.simr import simulation as sim

particles = sim.simulate_particles(model, sim.SimulatorConfig(n1=2000, n2=2000))
print("particles:", particles.shape)
print("the model's x, y, z, vx, vy, vz:     ", np.round(model[:6], 4))
print("SPAM's secondary at the end (time 0):", np.round(particles[-1], 4))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(*particles[:2000, :2].T, ",", color="C0"); ax.plot(*particles[2000:4000, :2].T, ",", color="C1")
ax.set_aspect("equal"); ax.set_xlabel("x (SPAM units, 15 kpc)"); ax.set_ylabel("y"); ax.set_title(f"SPAM's run of {TARGET}");

## A target's track
`encounter_rows(table, row)` gives the table rows of the target's encounter, earliest snapshot first. A target is one
row of its track: the rows before it are its past, the rows after it its future.

In [ ]:
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
rows = tng.encounter_rows(table, row)
now = int(np.flatnonzero(rows == row)[0])
print(type(rows).__name__, rows.dtype, rows.shape, "  the target is entry", now)
print("snapshots:", table["snap"][rows].tolist())
print(f"past: {now} snapshots back to {table['time_gyr'][row] - table['time_gyr'][rows[0]]:.2f} Gyr ago   "
      f"future: {len(rows) - 1 - now} snapshots, to {table['time_gyr'][rows[-1]] - table['time_gyr'][row]:.2f} Gyr on")

### Every track that holds a target
The 61 targets sit on 29 encounters: the same pair of galaxies taken as a target at several snapshots. Each line
below is one encounter's snapshots against the age of the universe; stars mark the snapshots that are targets, and a
cross marks a snapshot missing from the track.

In [ ]:
targets = sorted(int(path.stem.split("-")[-1]) for path in DATA.glob("target-archive-*.h5"))
tracks = {}
for target_id in targets:
    target_row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target_id)[0])
    tracks.setdefault(int(table["moi_2"][target_row]), []).append(target_row)
print(len(targets), "targets on", len(tracks), "encounters")

age = {int(snap): float(time) for snap, time in zip(table["snap"], table["time_gyr"])}   # snapshot -> Gyr
fig, ax = plt.subplots(figsize=(13, 7.5))
for line, (encounter, target_rows) in enumerate(sorted(tracks.items(), key=lambda item: table["snap"][item[1][0]])):
    rows = tng.encounter_rows(table, target_rows[0])
    snaps = table["snap"][rows]
    ax.plot(table["time_gyr"][rows], np.full(len(rows), line), ".-", color="C0", lw=0.8, ms=4)
    ax.plot(table["time_gyr"][target_rows], np.full(len(target_rows), line), "*", color="C3", ms=9)
    missing = sorted(set(range(snaps[0], snaps[-1])) - set(snaps.tolist()))
    known = [snap for snap in missing if snap in age]
    ax.plot([age[snap] for snap in known], np.full(len(known), line), "x", color="k", ms=7)
    ax.text(table["time_gyr"][rows[-1]] + 0.08, line, f"{encounter}  ({len(rows)} rows" + (f", missing {missing})" if missing else ")"),
            va="center", fontsize=7)
ax.set_xlabel("age of the universe (Gyr)"); ax.set_ylabel("encounter"); ax.set_yticks([])
ax.set_xlim(right=ax.get_xlim()[1] + 2.2); ax.set_title("the tracks: snapshots (dots), targets (stars), missing snapshots (crosses)");

`p` and `s` are the same two galaxies in every row of a track. TNG's merger tree says so: each galaxy's
`DescendantID` is the same letter's `SubhaloID` in the next snapshot's row.

In [ ]:
links = kept = 0
for encounter in np.unique(table["moi_2"]):
    rows = tng.encounter_rows(table, int(np.flatnonzero(table["moi_2"] == encounter)[0]))
    for here, after in zip(rows[:-1], rows[1:]):
        if table["snap"][after] == table["snap"][here] + 1:
            links += 1
            kept += all(table[f"{side}_DescendantID"][here] == table[f"{side}_SubhaloID"][after] for side in "ps")
print(f"{kept} of {links} snapshot-to-snapshot links carry p to p and s to s")

## The separation along a track
`track_separation(table, rows, primary_side)` gives the secondary's nucleus relative to the primary's at every row:
catalog `SubhaloPos`, made whole across the box edge, in physical kpc with each row's own scale factor. Still along
the box's axes. `primary_side` is one choice for the whole track, made at the target snapshot.

In [ ]:
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
rows = tng.encounter_rows(table, row)
separation = tng.track_separation(table, rows, "p")
distance = np.linalg.norm(separation, axis=1)
print(type(separation).__name__, separation.shape, separation.dtype)
print(f"{'snap':>4s} {'Gyr from target':>15s} {'a':>6s} {'x':>8s} {'y':>8s} {'z':>8s} {'distance (kpc)':>15s}")
for i, r in enumerate(rows):
    mark = "   <- the target" if r == row else ""
    print(f"{table['snap'][r]:4d} {table['time_gyr'][r] - table['time_gyr'][row]:15.2f} {table['scale_factor'][r]:6.3f} "
          f"{separation[i, 0]:8.1f} {separation[i, 1]:8.1f} {separation[i, 2]:8.1f} {distance[i]:15.1f}{mark}")

### Every track that holds a target
Distance between the nuclei against the age of the universe, one panel per encounter. Stars are the snapshots that
are targets. The dashed gray line is the same distance left comoving (not scaled by the scale factor), to show what
the scaling does.

In [ ]:
targets = sorted(int(path.stem.split("-")[-1]) for path in DATA.glob("target-archive-*.h5"))
tracks = {}
for target_id in targets:
    target_row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target_id)[0])
    tracks.setdefault(int(table["moi_2"][target_row]), []).append(target_row)

fig, axes = plt.subplots(5, 6, figsize=(19, 13), sharey=True)
summary = []
for ax, (encounter, target_rows) in zip(axes.flat, sorted(tracks.items(), key=lambda item: table["snap"][item[1][0]])):
    rows = tng.encounter_rows(table, target_rows[0])
    separation = tng.track_separation(table, rows, "p")
    distance, age = np.linalg.norm(separation, axis=1), table["time_gyr"][rows]
    ax.plot(age, distance / table["scale_factor"][rows], "--", color="gray", lw=0.8)
    ax.plot(age, distance, ".-", color="C0")
    here = np.isin(rows, target_rows)
    ax.plot(age[here], distance[here], "*", color="C3", ms=10)
    ax.set_title(str(encounter), fontsize=8); ax.set_yscale("log"); ax.grid(alpha=0.3)
    passes = int(((distance[1:-1] < distance[:-2]) & (distance[1:-1] < distance[2:])).sum())
    speed = np.linalg.norm(np.diff(separation, axis=0), axis=1) / np.diff(age) * 0.9778
    stored = np.linalg.norm(table["s_SubhaloPos"][rows].astype(np.float64) - table["p_SubhaloPos"][rows], axis=1)
    summary.append((encounter, len(rows), len(target_rows), distance[0], distance.min(), passes, speed.max(), stored.max() > 30000))
for ax in axes.flat[len(tracks):]:
    ax.axis("off")
fig.supxlabel("age of the universe (Gyr)"); fig.supylabel("distance between the nuclei (kpc)"); fig.tight_layout()

In [ ]:
print(f"{'encounter':15s} {'rows':>4s} {'targets':>7s} {'first (kpc)':>11s} {'closest':>8s} {'passes':>6s} {'fastest (km/s)':>14s}  across the box edge")
for encounter, n, held, first, closest, passes, fastest, across in summary:
    print(f"{encounter:<15d} {n:4d} {held:7d} {first:11.1f} {closest:8.1f} {passes:6d} {fastest:14.0f}  {'yes' if across else ''}")
print()
print("'passes': snapshots where the distance is smaller than at both neighbours. 'fastest': the largest change of the")
print("separation vector between consecutive snapshots, as a speed.")

## The track in SPAM's frame
The target's one rotation (`target["rotation"]`, from `measure_target`) turns the whole track into the frame SPAM runs
in. The target's own row lands on `(separation, 0, 0)`. The frame is fixed at the target snapshot, so rows further
back are not held in the x-y plane: the right panel shows how far out of it they sit.

In [ ]:
def track_in_frame(target_id, measured):
    # The track's rows, their time from the target (Gyr) and the secondary's path in SPAM's frame (kpc).
    target_row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target_id)[0])
    rows = tng.encounter_rows(table, target_row)
    path = tng.track_separation(table, rows, measured["primary_side"]) @ measured["rotation"].T
    return rows, table["time_gyr"][rows] - table["time_gyr"][target_row], path


def plot_track(target_id, measured, axes):
    rows, time, path = track_in_frame(target_id, measured)
    past, later, now = time <= 0, time > 0, time == 0
    for ax, (i, j) in zip(axes, ((0, 1), (0, 2))):
        ax.plot(path[past, i], path[past, j], ".-", color="C0", label="up to now")
        ax.plot(path[later | now, i], path[later | now, j], ".--", color="C0", mfc="none", label="later")
        ax.plot(0, 0, "k+", ms=12); ax.plot(path[now, i], path[now, j], "*", color="C3", ms=12, label="now")
        ax.set_aspect("equal", adjustable="datalim"); ax.set_xlabel("x (kpc)"); ax.set_ylabel("yz"[j - 1] + " (kpc)")
    axes[0].set_title(f"{target_id}: in the orbit's plane"); axes[1].set_title("edge-on: z is out of the plane")
    axes[0].legend(fontsize=8)
    distance = np.linalg.norm(path, axis=1)
    axes[2].plot(time, distance, ".-", color="C0", label="distance")
    axes[2].plot(time, np.abs(path[:, 2]), ".-", color="C1", label="|z|, out of the plane")
    axes[2].axvline(0, color="gray", lw=0.6); axes[2].set_xlabel("Gyr from the target snapshot"); axes[2].set_ylabel("kpc")
    axes[2].set_title("distance, and the part out of the plane"); axes[2].legend(fontsize=8)
    return path[now][0]


fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
print("the target's own row:", np.round(plot_track(TARGET, target, axes), 6), "  separation:", round(target["separation"], 6))

## The track's time axis
`track_time(table, rows, row)` gives each row's time relative to the target, in Gyr: 0 at the target and negative
before it, the same zero and sign as SPAM's orbit. The times come from each row's scale factor through
`age_of_universe`, TNG's cosmology in closed form.

In [ ]:
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
rows = tng.encounter_rows(table, row)
time = tng.track_time(table, rows, row)
print("age today:", round(float(tng.age_of_universe(1.0)), 3), "Gyr")
print("the table's time_gyr column against the formula, largest difference:",
      f"{np.abs(tng.age_of_universe(table['scale_factor']) - table['time_gyr']).max():.1e} Gyr")
print()
print("time from the target (Gyr):", np.round(time, 2))
print("in SPAM's units:           ", np.round(tng.spam_units(time, "time"), 1))

### The track and SPAM's orbit, same frame, units and clock
Everything below is in SPAM's units: lengths in 15 kpc, time in units of about 86.6 Myr. The track is the catalog's
snapshot positions turned into the target's frame; SPAM's orbit is `simulation.search_orbit` for the model
`spam_model(target)`, which runs from now back to t = -30 (dotted line). Track rows before that are further back than
SPAM looks.

In [ ]:
def compare_orbits(target_id, measured, axes):
    target_row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == target_id)[0])
    rows = tng.encounter_rows(table, target_row)
    path = tng.spam_units(tng.track_separation(table, rows, measured["primary_side"]) @ measured["rotation"].T, "length")
    time = tng.spam_units(tng.track_time(table, rows, target_row), "time")
    orbit = sim.search_orbit(tng.spam_model(measured))
    past, later = time <= 0, time >= 0
    axes[0].plot(orbit[:, 0], orbit[:, 1], color="C3", label="SPAM's orbit")
    axes[0].plot(path[past, 0], path[past, 1], ".-", color="C0", label="TNG track, up to now")
    axes[0].plot(path[later, 0], path[later, 1], ".--", color="C0", mfc="none", label="TNG track, later")
    axes[0].plot(0, 0, "k+", ms=12); axes[0].plot(path[time == 0, 0], path[time == 0, 1], "*", color="k", ms=12, label="now")
    axes[0].set_aspect("equal", adjustable="datalim"); axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
    axes[0].set_title(f"{target_id}: in the orbit's plane"); axes[0].legend(fontsize=7)
    axes[1].plot(orbit[:, 6], np.linalg.norm(orbit[:, :3], axis=1), color="C3", label="SPAM's orbit")
    axes[1].plot(time, np.linalg.norm(path, axis=1), ".-", color="C0", label="TNG track")
    axes[1].axvline(-30, color="gray", ls=":"); axes[1].axvline(0, color="gray", lw=0.6)
    axes[1].set_xlabel("time (SPAM units; 0 is now)"); axes[1].set_ylabel("distance between the nuclei")
    axes[1].set_title(f"{int((time < -30).sum())} of {len(time)} rows are before t = -30"); axes[1].legend(fontsize=7)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
compare_orbits(TARGET, target, axes)

## Errors
A wrong particle type or a missing file is named in the error.

In [ ]:
for call in (lambda: tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "wind"),
             lambda: tng.read_archive(DATA / "target-archive-1.h5"),
             lambda: tng.read_table(DATA / "nothing.h5")):
    try:
        call()
    except (ValueError, FileNotFoundError) as error:
        print(type(error).__name__ + ":", error)